In [0]:
dbutils.secrets.listScopes()

[SecretScope(name='kv-scope')]

In [0]:
key=dbutils.secrets.get(scope="kv-scope", key="container-secret")
print(key)

[REDACTED]


In [0]:
%sql
CREATE EXTERNAL LOCATION `main_store`
URL "abfss://main@amazonstore.dfs.core.windows.net/"
WITH (STORAGE CREDENTIAL `dipticred`)

### Create catalog and Schema

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS amazon_project;


In [0]:
%sql
USE CATALOG amazon_project;

CREATE SCHEMA IF NOT EXISTS bronze 
MANAGED LOCATION 'abfss://main@amazonstore.dfs.core.windows.net/bronze';

CREATE SCHEMA IF NOT EXISTS silver
MANAGED LOCATION 'abfss://main@amazonstore.dfs.core.windows.net/silver';

CREATE SCHEMA IF NOT EXISTS gold
MANAGED LOCATION 'abfss://main@amazonstore.dfs.core.windows.net/gold';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS metadeta
MANAGED LOCATION 'abfss://main@amazonstore.dfs.core.windows.net/logs';

In [0]:
%sql
-- Logging Table
CREATE TABLE IF NOT EXISTS metadeta.pipeline_logs (
    id STRING,
    step STRING,
    status STRING,
    message STRING,
    timestamp TIMESTAMP
);

### Configure path and

In [0]:
STORAGE_ACCOUNT = "amazonstore"
CONTAINER = "main"

CONTAINER_ROOT = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

# Folder Paths (based on your screenshot)
BRONZE_PATH = f"{CONTAINER_ROOT}bronze/"
SILVER_PATH = f"{CONTAINER_ROOT}silver/"
GOLD_PATH   = f"{CONTAINER_ROOT}gold/"
LOGS_PATH   = f"{CONTAINER_ROOT}logs/"

# If you have landing/raw files, use bronze as landing
LANDING_PATH = BRONZE_PATH

def write_log(step, status, message):
    """
    Writes logs into a Delta table: amazon_project.metadata.pipeline_logs
    """
    try:
        from pyspark.sql.functions import current_timestamp
        import uuid

        data = [(str(uuid.uuid4()), step, status, message)]

        df = spark.createDataFrame(
            data,
            ["id", "step", "status", "message"]
        ).withColumn("timestamp", current_timestamp())

        df.write.mode("append").saveAsTable(
            "amazon_project.metadata.pipeline_logs"
        )

    except Exception as e:
        print(f"Logging Error: {e}")

def get_latest_files():
    """
    Returns latest 2 file paths from bronze (landing zone)
    """
    try:
        files = dbutils.fs.ls(LANDING_PATH)

        if len(files) < 2:
            raise Exception("Not enough files in landing path")

        files = sorted(files, key=lambda x: x.modificationTime, reverse=True)

        return files[0].path, files[1].path

    except Exception as e:
        write_log("FILE_FETCH", "FAILED", str(e))
        raise